# Install Required Libraries

In [1]:
!pip install pdfplumber nltk
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\MSI\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

# Process All Constitution PDFs

This extracts text from each constitution and converts it into structured JSON.

In [4]:
# First, install required packages
!pip install pypdf2 PyMuPDF pdfplumber nltk

# Now run your script
import json
import os
import nltk
from nltk.tokenize import sent_tokenize

# Download required NLTK data
nltk.download('punkt_tab')
nltk.download('punkt')

INPUT_DIR = "/content/constitutions"
OUTPUT_JSON = "/content/processed_constitutions.json"

constitutions_data = {}

def extract_text(pdf_path):
    """Extract text from PDF using multiple methods"""
    text = ""

    # Method 1: Try PyMuPDF first (fitz)
    try:
        import fitz
        doc = fitz.open(pdf_path)
        for page in doc:
            text += page.get_text() + "\n"
        doc.close()
        print(f"  Used PyMuPDF (fitz) successfully")
        return text
    except Exception as e:
        print(f"  PyMuPDF failed: {e}")
        text = ""

    # Method 2: Try PyPDF2
    if not text:
        try:
            import PyPDF2
            with open(pdf_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                for page in reader.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
            print(f"  Used PyPDF2 successfully")
            return text
        except Exception as e:
            print(f"  PyPDF2 failed: {e}")
            text = ""

    # Method 3: Try pdfplumber with strict=False
    if not text:
        try:
            import pdfplumber
            with pdfplumber.open(pdf_path, strict=False) as pdf:
                for page in pdf.pages:
                    t = page.extract_text()
                    if t:
                        text += t + "\n"
            print(f"  Used pdfplumber successfully")
            return text
        except Exception as e:
            print(f"  pdfplumber failed: {e}")

    return text

# Process each PDF file
for file_name in os.listdir(INPUT_DIR):
    if file_name.endswith(".pdf"):
        full_path = os.path.join(INPUT_DIR, file_name)
        print(f"\nProcessing: {file_name}")

        try:
            raw_text = extract_text(full_path)

            if not raw_text or not raw_text.strip():
                print(f"  WARNING: No text extracted from {file_name}")
                raw_text = ""  # Set to empty string to avoid None

            # Tokenize sentences
            sentences = sent_tokenize(raw_text) if raw_text else []

            constitutions_data[file_name] = {
                "file_name": file_name,
                "raw_text": raw_text,
                "sentence_count": len(sentences),
                "sentences": sentences
            }

            print(f"  Success: Extracted {len(sentences)} sentences")

        except Exception as e:
            print(f"  ERROR processing {file_name}: {str(e)}")
            constitutions_data[file_name] = {
                "file_name": file_name,
                "error": str(e),
                "raw_text": "",
                "sentences": []
            }

# Save to JSON file
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(constitutions_data, f, indent=4, ensure_ascii=False)

print(f"\n{'='*50}")
print(f"PROCESSING COMPLETE")
print(f"{'='*50}")
print(f"Output saved to: {OUTPUT_JSON}")
print(f"Total files processed: {len(constitutions_data)}")

# Show summary
successful = sum(1 for data in constitutions_data.values() if data.get('sentence_count', 0) > 0)
failed = len(constitutions_data) - successful
print(f"Successful extractions: {successful}")
print(f"Failed extractions: {failed}")

# Show first file's structure if successful
if constitutions_data:
    first_key = list(constitutions_data.keys())[0]
    print(f"\nSample structure of first file ({first_key}):")
    print(f"  Sentence count: {constitutions_data[first_key].get('sentence_count', 0)}")
    if constitutions_data[first_key].get('sentences'):
        print(f"  First 2 sentences:")
        for i, sentence in enumerate(constitutions_data[first_key]['sentences'][:2], 1):
            print(f"    {i}. {sentence[:100]}...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 89.0 MB/s eta 0:00:00


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!



Processing: constitution-Latest.pdf
  Used PyMuPDF (fitz) successfully
  Success: Extracted 1983 sentences

Processing: Report-of-The-Donoughmore-Commission-gbdoed.pdf
  Used PyMuPDF (fitz) successfully
  Success: Extracted 0 sentences

Processing: 1978ConstitutionWithoutAmendments.pdf
  Used PyMuPDF (fitz) successfully
  Success: Extracted 1031 sentences

Processing: Report-of-The-Soulbury-Commission-hvpase.pdf
  Used PyMuPDF (fitz) successfully
  Success: Extracted 0 sentences

Processing: 1972-Constitution-of-Sri-Lanka-Article-105-–134-Chapter-XIII-dked1t.pdf
  Used PyMuPDF (fitz) successfully
  Success: Extracted 0 sentences

PROCESSING COMPLETE
Output saved to: /content/processed_constitutions.json
Total files processed: 5
Successful extractions: 2
Failed extractions: 3

Sample structure of first file (constitution-Latest.pdf):
  Sentence count: 1983
  First 2 sentences:
    1.   
The Constitution of the Democratic Socialist Republic of Sri Lanka 
 
THE CONSTITUTION 
 
 
OF TH...

# Extract Text from PDF

In [7]:
import pdfplumber

# Define the path to your PDF file
pdf_file = "/content/constitutions/constitution-Latest.pdf"  # Adjust this path

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

constitution_raw = extract_text_from_pdf(pdf_file)
print(f"Text length: {len(constitution_raw)} characters")
print(f"First 500 characters:\n{constitution_raw[:500]}...")

Text length: 448511 characters
First 500 characters:
The Constitution of the Democratic Socialist Republic of Sri Lanka
THE CONSTITUTION
OF THE
DEMOCRATIC SOCIALIST REPUBLIC
OF SRI LANKA
(As amended up to 31st October 2022)
Revised Edition – 2023
Published by the Parliament Secretariat
i i The Constitution of the Democratic Socialist Republic of Sri Lanka
The Constitution of the Democratic Socialist Republic of Sri Lanka
THE CONSTITUTION
OF THE
DEMOCRATIC SOCIALIST REPUBLIC
OF SRI LANKA
(As amended up to 31st October 2022)
Revised Edition – 2023
P...


# Clean Text (Remove headers, footers, weird spacing)

In [8]:
import re

def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"Page \d+", "", text)
    text = text.replace("—", "-")
    return text.strip()

constitution_clean = clean_text(constitution_raw)
constitution_clean[:2000]

'The Constitution of the Democratic Socialist Republic of Sri Lanka THE CONSTITUTION OF THE DEMOCRATIC SOCIALIST REPUBLIC OF SRI LANKA (As amended up to 31st October 2022) Revised Edition – 2023 Published by the Parliament Secretariat i i The Constitution of the Democratic Socialist Republic of Sri Lanka The Constitution of the Democratic Socialist Republic of Sri Lanka THE CONSTITUTION OF THE DEMOCRATIC SOCIALIST REPUBLIC OF SRI LANKA (As amended up to 31st October 2022) Revised Edition – 2023 Published by the Parliament Secretariat Printed at the Department of Government Printing ii The Constitution of the Democratic Socialist Republic of Sri Lanka Revised Edition – 2023 Published by the Parliament Secretariat The Constitution of the Democratic Socialist Republic of Sri Lanka iii This unofficial edition edited by the Bills Office of the Legislative Services Department of Parliament of Sri Lanka reproduces the text of the Constitution of the Democratic Socialist Republic of Sri Lanka 

# Extract Articles (Chapter III – Fundamental Rights)

In [9]:
import re

def split_articles(text):
    pattern = r"(Article\s+(10|11|12|13|14|15|16|17|18))"
    parts = re.split(pattern, text)

    articles = {}
    current_key = None
    buffer = ""

    for part in parts:
        if part.startswith("Article"):
            if current_key and buffer.strip():
                articles[current_key] = buffer.strip()
            current_key = part.replace(" ", "").strip()  # e.g., Article10
            buffer = ""
        else:
            buffer += " " + part

    if current_key and buffer.strip():
        articles[current_key] = buffer.strip()

    return articles

fundamental_rights = split_articles(constitution_clean)

fundamental_rights

{'Article11': '11 1M], means any person who holds office as - 181 - Substituted by the Seventeenth Amendment to the Constitution Sec. 22(1). The Constitution of the Democratic Socialist Republic of Sri Lanka 229 (a) a Judge of the Supreme Court or a Judge of the Court of Appeal; (b) any Judge of the High Court or any Judge, presiding officer or member of any other Court of First Instance, tribunal or institution created and established for the administration of Justice or for the adjudication of any labour or other dispute but does not include a person who performs arbitral functions or a public officer whose principal duty or duties is or are not the performance of functions of a judicial nature. No court or tribunal or institution shall have jurisdiction to determine the question whether a person is a judicial officer within the meaning of the Constitution but such question shall be determined by the Judicial Service Commission whose decision thereon shall be final and conclusive. No

# Convert Articles into Rich JSON Format

This JSON will be used by your backend for rights detection.

In [10]:
import json

articles_json = {}

for key, text in fundamental_rights.items():
    article_num = int(key.replace("Article", ""))

    articles_json[article_num] = {
        "article_number": article_num,
        "text": text,
        "keywords": list(set(re.findall(r"[A-Za-z]+", text))),
    }

articles_json

{11: {'article_number': 11,
  'text': '11 1M], means any person who holds office as - 181 - Substituted by the Seventeenth Amendment to the Constitution Sec. 22(1). The Constitution of the Democratic Socialist Republic of Sri Lanka 229 (a) a Judge of the Supreme Court or a Judge of the Court of Appeal; (b) any Judge of the High Court or any Judge, presiding officer or member of any other Court of First Instance, tribunal or institution created and established for the administration of Justice or for the adjudication of any labour or other dispute but does not include a person who performs arbitral functions or a public officer whose principal duty or duties is or are not the performance of functions of a judicial nature. No court or tribunal or institution shall have jurisdiction to determine the question whether a person is a judicial officer within the meaning of the Constitution but such question shall be determined by the Judicial Service Commission whose decision thereon shall be 

# Save JSON File

This file goes into:

/data/sri_lanka_legal_corpus/constitutional_articles_mapping.json

In [12]:
from google.colab import files

output_file = "constitution_articles.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(articles_json, f, indent=4, ensure_ascii=False)

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Generate Enriched Fundamental Rights Regex Patterns (IMPROVED)

These patterns are FAR more accurate and will detect rights phrases even if written differently.

In [13]:
enriched_patterns = {
    10: r"(freedom of thought|conscience|religion|worship)",
    11: r"(torture|inhuman treatment|degrading punishment)",
    12: r"(equality|equal protection|discrimination|equal before the law)",
    13: r"(arrest|detention|remand|custody|criminal charge)",
    14: r"(speech|expression|peaceful assembly|association|union|movement|residence|occupation)",
    15: r"(freedom of movement|freedom to reside)",
    16: r"(language rights|official language|national language|Sinhala|Tamil)",
    17: r"(procedures for rights enforcement|FR application|Article 126)",
    18: r"(privacy|family|home|correspondence)"
}

with open("enriched_fundamental_rights_patterns.json", "w", encoding="utf-8") as f:
    json.dump(enriched_patterns, f, indent=4)

files.download("enriched_fundamental_rights_patterns.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>